In [221]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

In [222]:
data= pd.read_csv("/content/oulad_flat.csv")

In [223]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   code_module            32593 non-null  object 
 1   code_presentation      32593 non-null  object 
 2   id_student             32593 non-null  int64  
 3   gender                 32593 non-null  object 
 4   region                 32593 non-null  object 
 5   highest_education      32593 non-null  object 
 6   imd_band               31482 non-null  object 
 7   age_band               32593 non-null  object 
 8   num_of_prev_attempts   32593 non-null  int64  
 9   studied_credits        32593 non-null  int64  
 10  disability             32593 non-null  object 
 11  final_result           32593 non-null  object 
 12  date_registration      32548 non-null  float64
 13  date_unregistration    10072 non-null  float64
 14  assess_events          32593 non-null  float64
 15  as

In [224]:
data.isnull().sum()

,0
code_module,0
code_presentation,0
id_student,0
gender,0
region,0
highest_education,0
imd_band,1111
age_band,0
num_of_prev_attempts,0
studied_credits,0


In [225]:
df = data.copy()

# Target
df["withdraw"] = (df["final_result"] == "Withdrawn").astype(int)

# Drop some columns
cols = [
    "final_result",
    "date_unregistration",
    "registration_duration",
    "code_presentation",
   # "date_registration",
    "code_module",
    "imd_band",
    "id_student",
    "region",
    "first_activity_day",
    "last_activity_day",
     "max_daily_clicks",
    "engagement_span"


]

df = df.drop(columns=[c for c in cols if c in df.columns])


In [226]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   gender                 32593 non-null  object 
 1   highest_education      32593 non-null  object 
 2   age_band               32593 non-null  object 
 3   num_of_prev_attempts   32593 non-null  int64  
 4   studied_credits        32593 non-null  int64  
 5   disability             32593 non-null  object 
 6   date_registration      32548 non-null  float64
 7   assess_events          32593 non-null  float64
 8   assess_submissions     32593 non-null  float64
 9   late_submissions       32593 non-null  float64
 10  banked_count           32593 non-null  float64
 11  avg_score              32593 non-null  float64
 12  sum_score              32593 non-null  float64
 13  max_score              32593 non-null  float64
 14  min_score              32593 non-null  float64
 15  vl

In [227]:
df.dropna(inplace=True)

In [228]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32548 entries, 0 to 32592
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   gender                 32548 non-null  object 
 1   highest_education      32548 non-null  object 
 2   age_band               32548 non-null  object 
 3   num_of_prev_attempts   32548 non-null  int64  
 4   studied_credits        32548 non-null  int64  
 5   disability             32548 non-null  object 
 6   date_registration      32548 non-null  float64
 7   assess_events          32548 non-null  float64
 8   assess_submissions     32548 non-null  float64
 9   late_submissions       32548 non-null  float64
 10  banked_count           32548 non-null  float64
 11  avg_score              32548 non-null  float64
 12  sum_score              32548 non-null  float64
 13  max_score              32548 non-null  float64
 14  min_score              32548 non-null  float64
 15  vle_eve

In [229]:
df['withdraw'].value_counts()

,count
withdraw,
0,22431
1,10117


In [230]:
X = df.drop(columns=["withdraw"])
y = df["withdraw"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [231]:
numerical = [
 'num_of_prev_attempts', 'studied_credits',
 'date_registration', 'assess_events', 'assess_submissions',
 'late_submissions', 'banked_count', 'avg_score', 'sum_score',
 'max_score', 'min_score', 'vle_events', 'total_clicks',
 'active_days', 'late_ratio', 'clicks_per_active_day'
]


ordinal = [
    "age_band",
    "highest_education"
]

nominal= [
    "gender",
    "disability"
]


In [232]:
age_order = [
    "0-35", "35-55", "55<="
]

education_order = [
    "No Formal quals",
    "Lower Than A Level",
    "A Level or Equivalent",
    "HE Qualification",
    "Post Graduate Qualification"
]


In [233]:
num_pipeline = Pipeline([
    # ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [234]:
ord_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OrdinalEncoder(
        categories=[age_order, education_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])


In [235]:
nom_pipeline = Pipeline([
    # ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [236]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numerical),
        ("ord", ord_pipeline, ordinal),
        ("nom", nom_pipeline, nominal)
    ]
)


In [237]:
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre= preprocessor.transform(X_test)


In [238]:
models = {
    "XGB": XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )
}


'LogReg': np.float64(0.9208460246987114)

 'RF': np.float64(0.9266353909131438)

 'XGB': np.float64(0.9294618170892667)


In [239]:
results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)
    results[name] = auc

print(results)

{'LogReg': np.float64(0.9208460246987114), 'RF': np.float64(0.9266353909131438), 'XGB': np.float64(0.9294618170892667)}


In [240]:
print(classification_report(y_test, pipe.predict(X_test)))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4486
           1       0.75      0.79      0.77      2024

    accuracy                           0.85      6510
   macro avg       0.83      0.84      0.83      6510
weighted avg       0.86      0.85      0.86      6510

